In [12]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorboard.plugins.hparams import api as hp

#### Downloading and Preprocessing the data

In [13]:
BUFFER_SIZE = 70000
BATCH_SIZE = 128
NUM_EPOCHS = 20

In [14]:
mnist_dataset, mnist_info = tfds.load(name = 'mnist', with_info = True, as_supervised=True)

In [15]:
mnist_train, mnist_test = mnist_dataset['train'], mnist_dataset['test']

In [16]:
def scale(image,label):
    image = tf.cast(image,tf.float32)
    image /= 255.
    return image,label


In [17]:
train_and_validation_data = mnist_train.map(scale)
test_data = mnist_test.map(scale)

In [18]:
num_validation_samples = 0.1 * mnist_info.splits['train'].num_examples
num_validation_samples = tf.cast(num_validation_samples,tf.int64)
num_test_samples = mnist_info.splits['test'].num_examples
num_test_samples = tf.cast(num_test_samples,tf.int64)

In [19]:
train_and_validation_data = train_and_validation_data.shuffle(BUFFER_SIZE)
train_data = train_and_validation_data.skip(num_validation_samples)
validation_data = train_and_validation_data.take(num_validation_samples)

In [20]:
train_data = train_data.batch(BATCH_SIZE)

In [21]:
validation_data = validation_data.batch(num_validation_samples)
test_data = test_data.batch(num_test_samples)

### Defining Hyperparameters

In [22]:
HP_FILTER_SIZE = hp.HParam('filter_size',hp.Discrete([3,5,7]))
HP_OPTIMIZER = hp.HParam('optimizer',hp.Discrete(['adam','sgd']))

METRIC_ACCURACY = 'accuracy'

with tf.summary.create_file_writer('logs/hparam_tuning').as_default():
    hp.hparams_config(
        hparams= [HP_FILTER_SIZE,HP_OPTIMIZER],
        metrics= [hp.Metric(METRIC_ACCURACY,display_name='Accuracy')],
    )

#### Create the model and train it

In [23]:
def train_test_model(hparams):
    model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(50,hparams[HP_FILTER_SIZE],activation='relu',input_shape=(28,28,1)),
    tf.keras.layers.MaxPooling2D(pool_size=(2,2)),
    tf.keras.layers.Conv2D(50,hparams[HP_FILTER_SIZE],activation='relu'),
    tf.keras.layers.MaxPooling2D(pool_size=(2,2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(10)
    ])
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    model.compile(optimizer=hparams[HP_OPTIMIZER],loss=loss_fn,metrics=['accuracy'])
    early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    mode='auto',
    min_delta=0,
    patience=2,
    verbose=0,
    restore_best_weights=True
    )
    model.fit(
    train_data,
    epochs = NUM_EPOCHS,
    callbacks = [early_stopping],
    validation_data=validation_data,
    verbose =2 #this specifies what to print out during the training process. verbose = 2 means it will print info only at the end 
    # of each epoch while verbose = 1 displays progress bar for every batch
    )
    _,accuracy = model.evaluate(test_data)
    return accuracy

In [24]:
def run(log_dir,hparams):
    with tf.summary.create_file_writer(log_dir).as_default():
        hp.hparams(hparams) # record the values used in this trial 
        accuracy = train_test_model(hparams)
        tf.summary.scalar(METRIC_ACCURACY,accuracy,step=1)

#### Training the model with different hyperparameters

In [25]:
session_num = 0

for filter_size in HP_FILTER_SIZE.domain.values:
    for optimizer in HP_OPTIMIZER.domain.values:
        hparams = {
            HP_FILTER_SIZE: filter_size,
            HP_OPTIMIZER: optimizer
        }
        run_name = 'run-%d' %session_num
        print('--- starting trial: %s' %run_name)
        print({h.name: hparams[h] for h in hparams})
        run('logs/hparam_tuning/'+run_name,hparams)

        session_num += 1

--- starting trial: run-0
{'filter_size': 3, 'optimizer': 'adam'}
Epoch 1/20


/Users/ankit/Workspace/Projects/ankit-github/AI-ML-DS/53.Convolutional-Neural-Networks-with-TensorFlow-in-Python/.venv/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/Users/ankit/Workspace/Projects/ankit-github/AI-ML-DS/53.Convolutional-Neural-Networks-with-TensorFlow-in-Python/.venv/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1787198922.314265 1433081 tf_record_dataset_op.cc:396] The default buffer size is 262144, which is o

422/422 - 8s - 18ms/step - accuracy: 0.9195 - loss: 0.2867 - val_accuracy: 0.9708 - val_loss: 0.0863
Epoch 2/20
422/422 - 7s - 17ms/step - accuracy: 0.9772 - loss: 0.0751 - val_accuracy: 0.9812 - val_loss: 0.0600
Epoch 3/20
422/422 - 7s - 17ms/step - accuracy: 0.9827 - loss: 0.0564 - val_accuracy: 0.9858 - val_loss: 0.0437
Epoch 4/20
422/422 - 7s - 18ms/step - accuracy: 0.9857 - loss: 0.0470 - val_accuracy: 0.9898 - val_loss: 0.0388
Epoch 5/20
422/422 - 8s - 18ms/step - accuracy: 0.9880 - loss: 0.0396 - val_accuracy: 0.9900 - val_loss: 0.0317
Epoch 6/20
422/422 - 8s - 18ms/step - accuracy: 0.9894 - loss: 0.0341 - val_accuracy: 0.9915 - val_loss: 0.0346
Epoch 7/20
422/422 - 8s - 18ms/step - accuracy: 0.9907 - loss: 0.0302 - val_accuracy: 0.9930 - val_loss: 0.0261
Epoch 8/20
422/422 - 8s - 19ms/step - accuracy: 0.9914 - loss: 0.0268 - val_accuracy: 0.9933 - val_loss: 0.0230
Epoch 9/20
422/422 - 8s - 19ms/step - accuracy: 0.9925 - loss: 0.0240 - val_accuracy: 0.9948 - val_loss: 0.0168
Epo

### Visualizing in Tensorboard

**Notes:**

- The %tensorboard magic is idempotent per logdir — re-running the cell reuses this same server rather than starting a new one. If you want a fresh one: %tensorboard --logdir logs/hparam_tuning --port 6007.
- If you'd rather not depend on the magic at all, run it from a terminal in the notebook's folder: .venv/bin/tensorboard --logdir logs/hparam_tuning.

Just open http://localhost:6006 in your browser and click the HPARAMS tab.

In [27]:
%load_ext tensorboard
%tensorboard --logdir "logs/hparam_tuning"

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard
